### <b>PERFORMANCE GAIN IN SQL GENERATION LEVERAGING MULTI AGENT COLLABORATION</b>


1. Add model configs
- Instruction models with RAG aided
- Models aided with tool calls
- Natural-sql, small size sql model
- 

In [ ]:
# SQL Expert and Tool Executor Model Configs
tool_executor_config = {
    "model": "mistral-nemo:latest",
    "base_url": "http://127.0.0.1:11435/v1",
    "api_key":"placeholder",
}

sql_expert1_config = {
    "model": "mCodeLlama-13B-Instruct:latest",
    "base_url": "http://127.0.0.1:11435/v1",
    "api_key":"placeholder",
}
sql_expert2_config = {
    "model": "Qwen2.5-7B-Instruct:latest",
    "base_url": "http://127.0.0.1:11435/v1",
    "api_key":"placeholder",
}

## Import packages

In [ ]:
import os
import openai
from typing_extensions import Annotated
from rich.markdown import Markdown

from pydantic import BaseModel

from autogen_agentchat.agents import AssistantAgent

from autogen_core.models import (
    LLMMessage,
    ModelFamily,
    ModelInfo,
)
from autogen_ext.models.openai import OpenAIChatCompletionClient

from autogen_core.tools import FunctionTool

from sqlalchemy import create_engine
from llama_index.tools.database import DatabaseToolSpec
from llama_index.core.utilities.sql_wrapper import SQLDatabase

from autogen_agentchat.ui import Console
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex

from pathlib import Path
from llama_index.core.schema import TextNode
from sqlalchemy.dialects.sqlite.base import SQLiteDialect
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI


In [3]:
import os
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import VectorIndexAutoRetriever
from llama_index.core.vector_stores import MetadataInfo, VectorStoreInfo
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.ollama import Ollama


DB_PATH="/home/simges/.cache/spider/test_database"
SENTENCE_TRANSFORMERS_HOME="/home/simges/.cache/huggingface/hub/sentence-transformers/{model_name}"

MINILM_EMBEDDING_MODEL_PATH=SENTENCE_TRANSFORMERS_HOME.format(model_name="/all-MiniLM-L6-v2")

os.environ["TIKTOKEN_CACHE_DIR"]="/home/simges/cl100k_base/"

minilm_embed = HuggingFaceEmbedding(model_name=MINILM_EMBEDDING_MODEL_PATH)
from llama_index.core import Settings

Settings.embed_model = HuggingFaceEmbedding(model_name=MINILM_EMBEDDING_MODEL_PATH)
Settings.llm = Ollama(model="mistral-nemo", request_timeout=360.0)

In [4]:
import time

# Load documents and build index
documents = SimpleDirectoryReader("./sql-best-practices").load_data()
vector_index = VectorStoreIndex.from_documents(documents)

vector_store_info = VectorStoreInfo(
    content_info="sql best practices",
    metadata_info=[
        MetadataInfo(
            name="syntax practices", type="str", description="supportive documents"
        ),
    ],
)
vector_auto_retriever = VectorIndexAutoRetriever(
    vector_index, vector_store_info=vector_store_info
)

retriever_query_engine = RetrieverQueryEngine.from_args(vector_auto_retriever)

In [5]:
time.sleep(5)
response = retriever_query_engine.query("""
SELECT orders.id as myorder, customers.name, orders.total_price
FROM orders
JOIN customers ON orders.customer_id = customers.id
WHERE orders.status = 'completed'
AND (orders.total_price > 100 OR orders.total_price = 100)
ORDER BY orders.total_price ASC;
""")

print(str(response))

This SQL query retrieves the order ID and associated customer name, along with the total price of completed orders where the total price is greater than or equal to 100. The results are ordered by total price in ascending order.


## CREATE FUNCTION TOOLS

### 1. **Syntax Checker Tool**
Parse sql query through an open source sqlglot tool,
-   If error, the tool shall return false, otherwise true.
-   Error message shall be fed to LLM.

### 2. **Database Spec Tool**
Map entities and attributes to the database tables & columns, perform schema linking.
-   If error, the tool shall return false, otherwise true.
-   Error message shall be fed to LLM.

In [3]:
# foreign keys and tables of database
import csv
csv.field_size_limit(10000000)

db_entities={}
database_index={}
dbs={}

test_dbs = ["concert_singer", "pets_1", "car_1", "flight_2", "employee_hire_evaluation", "cre_Doc_Template_Mgt", 
            "course_teach", "museum_visit", "battle_death", "student_transcripts_tracking", "tvshow", 
            "poker_player", "orchestra", "network_1", "dog_kennels", "singer", "real_estate_properties", "wta_1"]

for db_id in test_dbs:
    db_engine = create_engine(f"sqlite:///{DB_PATH}/{db_id}/{db_id}.sqlite")
    print(db_id)
    dbs[db_id] = SQLDatabase(engine=db_engine)
    schema = DatabaseToolSpec(sql_database=dbs[db_id]).to_tool_list()[1].call().raw_output

    table_names = DatabaseToolSpec(sql_database=dbs[db_id]).to_tool_list()[2].call().raw_output
    db_entities[db_id]={"foreign_keys":"", "tables": {}}
    for table_name in table_names:
        db_entities[db_id]["tables"][table_name]=dbs[db_id].get_single_table_info(table_name) + "\n"
        
        fkeys = dbs[db_id].get_foreign_keys(dialect=SQLiteDialect(), table_name=table_name)
        if isinstance(fkeys, list) and fkeys:  # Ensure it's a non-empty list    
            for fk in fkeys:
                constrained_columns = fk["constrained_columns"]
                referred_table = fk["referred_table"]
                referred_columns = fk["referred_columns"]
                db_entities[db_id]["foreign_keys"] += f"referring column {table_name}.{constrained_columns} to referred column {referred_table}.{referred_columns}\n"

NameError: name 'create_engine' is not defined

In [ ]:

syntax_fixer_message = """Remove aliasing (AS) for columns: {sql} 
Remove aliasing (AS) for aggregation columns: {sql} 
Optimize it based on the rules below: "

### Common SQL Syntax Errors and Tips for Correction:
1. Missing or Incorrect Aliases

Error: Forgetting to provide aliases when using multiple tables in joins.

Correction: Always use aliases for tables when joining tables.

SELECT o.order_id, c.customer_name 
FROM orders AS o
JOIN customers AS c ON o.customer_id = c.customer_id;

2. Incorrect Use of WHERE vs. HAVING

Error: Using WHERE for filtering aggregated data.

Correction: Use WHERE for non-aggregated conditions and HAVING for conditions on aggregated data (like SUM, COUNT, etc.).

-- Incorrect:
SELECT department, COUNT(*) 
FROM employees 
WHERE COUNT(*) > 10;

-- Correct:
SELECT department, COUNT(*) 
FROM employees 
GROUP BY department 
HAVING COUNT(*) > 10;

3. Using SELECT * Without Need

Error: Using SELECT * when you only need specific columns.

Correction: Always specify the exact columns you need to retrieve to reduce unnecessary data retrieval and improve performance.

-- Instead of:
SELECT * FROM employees;

-- Use:
SELECT employee_id, name, department FROM employees;

4. Improper Use of JOIN Conditions

Error: Missing or incorrect join conditions, leading to Cartesian products (unwanted rows).

Correction: Always define proper join conditions using ON and ensure that the correct columns are being compared.

-- Incorrect:
SELECT orders.order_id, customers.customer_name 
FROM orders 
JOIN customers;

-- Correct:
SELECT orders.order_id, customers.customer_name 
FROM orders 
JOIN customers ON orders.customer_id = customers.customer_id;

5. Wrong Placement of ORDER BY and GROUP BY Clauses

Error: Using ORDER BY before GROUP BY in a query that includes both.

Correction: GROUP BY must come before ORDER BY in SQL query structure.

-- Incorrect:
SELECT department, COUNT(*) 
FROM employees 
ORDER BY department 
GROUP BY department;

-- Correct:
SELECT department, COUNT(*) 
FROM employees 
GROUP BY department 
ORDER BY department;

6. Missing or Extra Parentheses in Expressions

Error: Forgetting or incorrectly placing parentheses around expressions, especially in CASE, JOIN, or WHERE clauses.

Correction: Always check that parentheses are balanced and correctly placed.

-- Incorrect:
SELECT name FROM employees WHERE (salary > 50000 OR department_id = 2;

-- Correct:
SELECT name FROM employees WHERE (salary > 50000 OR department_id = 2);

7. Using AND and OR Without Proper Parentheses

Error: Using AND and OR in the same condition without proper parentheses, leading to ambiguous logic.

Correction: Use parentheses to clarify the logic when combining AND and OR.

-- Incorrect:
SELECT * FROM orders WHERE status = 'completed' OR status = 'pending' AND order_date > '2025-01-01';

-- Correct:
SELECT * FROM orders WHERE (status = 'completed' OR status = 'pending') AND order_date > '2025-01-01';
"""

g_dbid=""
g_question=""
### 1.1 Create Tools
# a. SQL executor
async def execute_query(sql: Annotated[str, "sql query"]) -> Annotated[str, "query results"]:
    try:
        execution_results = dbs[g_dbid].run_sql(sql)[0]
        return { "question": g_question,
                 "db_id": g_dbid,
                 "status": "success"}
    except Exception as e:
        return { "question": g_question,
                 "db_id": g_dbid,
                 "status": "failure: " + str(e) }
execute_query_tool = FunctionTool(execute_query, description="Execute sql query.")


async def prompt_maker(sql: Annotated[str, "sql query"]) -> Annotated[str, "query results"]:
    return syntax_fixer_message.format(sql=sql)
prompt_maker_tool = FunctionTool(prompt_maker, description="Generate sql fixer prompt.")

# %% [markdown]
# ## Runtime
# - Start runtime
# - Register tools to runtime

# %% [markdown]
# ## 1. Agent Prompts

# %%
tool_client_prompt = """You are an AI assistant with access to {tool} tool. 
Whenever the user provides an SQL query, you must invoke {tool} tool with the provided information. Follow these rules:

Extract necessary details from the user’s input.
Call the tool with the extracted data in the correct format.
Do not answer manually.

Your Task:

Identify the correct tool to use.
Call it with the required arguments.
Return the tool's response to the user."""


schema_linker_prompt = """
You are an expert in extracting key components from natural language expressions. Your task is to identify and classify the following entities: tables (objects/entities from which data is fetched), columns (fields/attributes of those tables), and conditions (filters or constraints). Return a structured output with identified tables, columns, and conditions.
Avoid aliasing for columns and aggregations.

1. tables: student

2. columns: student.age
   
3. conditions: student.age < 19

4. joins: students.student_id = courses.student_id

"""

sql_constructor_prompt = """Construct the SQL query based on the information given.
Here are some tips for generating SQL queries that focus on performance, syntax, and simplicity:

1. Performance Tips
Use Proper Indexes: Ensure that the columns used in WHERE, JOIN, ORDER BY, and GROUP BY clauses are indexed. This speeds up lookups and reduces the query execution time.

a. Avoid SELECT * : Only select the columns you need. Using SELECT * fetches all columns, which can result in unnecessary data transfer and slower performance, especially with large tables.

-- Instead of:
SELECT * FROM orders;

-- Use:
SELECT order_id, customer_id, order_date FROM orders;

b. Limit Rows with WHERE: Always filter data as early as possible using the WHERE clause to reduce the number of rows processed, especially for large datasets.

SELECT * FROM orders WHERE order_date > '2025-01-01';

c. Use JOIN Instead of OUTER JOIN: If you only need matching rows, use JOIN. LEFT JOIN and RIGHT JOIN can be slower because they return unmatched rows with NULL values.

d. Avoid Multiple Subqueries: Try to use JOIN instead of nested subqueries, especially if those subqueries are in the WHERE clause. Subqueries can be less efficient because they require multiple passes through the data.

-- Instead of this:
SELECT * FROM orders WHERE order_id IN (SELECT order_id FROM order_details);

-- Use this:
SELECT orders.* FROM orders INNER JOIN order_details ON orders.order_id = order_details.order_id;

2. Syntax Tips
a. Use Proper Aliases with the AS Keyword: Always use aliases for tables and columns when necessary for clarity, especially when working with multiple tables or complex queries.

SELECT T1.order_id, T2.customer_name
FROM orders AS T1
JOIN customers AS T2 ON T1.customer_id = T2.customer_id;

b. Avoid Ambiguity in Joins: Always specify table names (or aliases) when referring to columns in JOIN queries to avoid ambiguity.

SELECT orders.order_id, customers.customer_name 
FROM orders 
JOIN customers ON orders.customer_id = customers.customer_id;

3. Simplicity Tips
a. Avoid Over-Complex Queries: Break down complex queries into simpler steps where possible. For example, use common table expressions (CTEs) or temporary tables for intermediate results.

WITH recent_orders AS (
  SELECT * FROM orders WHERE order_date > '2025-01-01'
)
SELECT * FROM recent_orders WHERE customer_id = 101;

b. Limit Query Scope: If possible and asked in the question, always limit the rows returned (e.g., using LIMIT or TOP) to avoid unnecessarily large result sets.
"""

join_expert_prompt = """ Please ensure that join operations in SQL queries are used correctly based on the specific needs of the query. Here's a quick reminder about the common join operators:
1. JOIN
Use it when: You only need to return rows that have corresponding matches in both tables.

Scenario: This is the most common join type. It is efficient when you want to combine data from two or more tables where there is a direct relationship, and you're only interested in records that exist in both tables.

Performance: Generally, JOIN is the fastest type of join because it only returns rows that meet the join condition, eliminating unnecessary rows.

2. LEFT JOIN (LEFT OUTER JOIN)
Use it when: You want to return all rows from the left table, even if there is no match in the right table.

Scenario: Use this when you need to include all records from one table, regardless of whether a related record exists in the other table. This is useful in scenarios where the existence of the left table's data is more important than the right table's.

Performance: LEFT JOIN is usually less efficient than INNER JOIN because it has to check for the absence of matches and return NULL values when there is no match.

3. RIGHT JOIN (RIGHT OUTER JOIN)
Use it when: You want to return all rows from the right table, even if there is no match in the left table.

Scenario: RIGHT JOIN is essentially the reverse of LEFT JOIN. You'd use it when the right table is considered the "primary" table and you want to keep all records from it, regardless of whether there's a match in the left table.

Performance: Like LEFT JOIN, RIGHT JOIN can be less performant than INNER JOIN, especially on larger datasets, due to the inclusion of unmatched rows from the right table.

4. FULL JOIN (FULL OUTER JOIN)
Use it when: You need to return all rows when there is a match in either the left or right table.

Scenario: This join type is used when you want to ensure that no rows are omitted, whether they have a match in both tables or not. It returns every row from both tables, with NULL where data from one table doesn't have a corresponding match in the other.

Performance: FULL JOIN can be the least performant because it combines all rows from both tables and has to check for matches, filling gaps with NULL where no match exists.

When to Avoid Using Joins:
Too many joins: If you're joining too many tables, especially with OUTER JOINS, it could significantly degrade query performance. Always evaluate whether all the joins are necessary.

Joins on large datasets: For large tables, consider using indexes on columns involved in the join conditions to optimize performance.

Incorrect use of OUTER JOINs: When data should be filtered, not all OUTER JOINs are appropriate. Overuse can lead to unnecessary data being included.

General Guidelines for Choosing Joins:
Data Requirements: Determine whether you need all data from one table (use LEFT JOIN, RIGHT JOIN, or FULL JOIN), or just the matching data (use JOIN).

Query Complexity: Simplify your queries when possible. The more joins you have, the more complex the query becomes, and the slower it could execute.

Performance Considerations: Always prefer JOIN over OUTER JOIN when you do not need unmatched rows. Ensure that the columns involved in joins are indexed for faster lookups.

Ensuring correct order of joins to improve query performance.
"""

# TOOL EXECUTOR CLIENT 
tool_executor_client = OpenAIChatCompletionClient(model=tool_executor_config["model"],
    base_url=tool_executor_config["base_url"],
    api_key=tool_executor_config["api_key"],
    model_info = ModelInfo(
        vision=False, function_calling=True,
        json_output=True, family=ModelFamily.UNKNOWN
    ),
    temperature=0.0,
    max_tokens=128000,
)

# %%
# SQL EDITORS
class SchemaLinkerOutput(BaseModel):
    tables: str
    columns: str
    conditions: str

schema_linker_client = OpenAIChatCompletionClient(
        model=sql_expert3_config["model"],
        base_url=sql_expert3_config["base_url"],
        api_key=sql_expert3_config["api_key"],
        model_info = ModelInfo(
            vision=False, function_calling=True,
            json_output=True, family=ModelFamily.UNKNOWN
        ),
        temperature=0.0,
        response_format=SchemaLinkerOutput,
        max_tokens=128000,
)

# CONSTRUCTOR CLIENT
class SQLConstructorOutput(BaseModel):
    class Step(BaseModel):
        explanation: str
    steps: list[Step]
    sql: str

sql_constructor_client = OpenAIChatCompletionClient(
        model=sql_expert2_config["model"],
        base_url=sql_expert2_config["base_url"],
        api_key=sql_expert2_config["api_key"],
        model_info = ModelInfo(
            vision=False, function_calling=True,
            json_output=True, family=ModelFamily.UNKNOWN
        ),
        temperature=0.0,
        response_format=SQLConstructorOutput,
        max_tokens=128000,
)

class SQLOutput(BaseModel):
    sql: str

join_expert_client = OpenAIChatCompletionClient(
        model=sql_expert1_config["model"],
        base_url=sql_expert1_config["base_url"],
        api_key=sql_expert1_config["api_key"],
        model_info = ModelInfo(
            vision=False, function_calling=True,
            json_output=True, family=ModelFamily.UNKNOWN
        ),
        temperature=0.0,
        response_format=SQLOutput,
        max_tokens=128000,
)

syntax_fixer_client = OpenAIChatCompletionClient(
        model=sql_expert4_config["model"],
        base_url=sql_expert4_config["base_url"],
        api_key=sql_expert4_config["api_key"],
        model_info = ModelInfo(
            vision=False, function_calling=True,
            json_output=True, family=ModelFamily.UNKNOWN
        ),
        temperature=0.0,
        response_format=SQLOutput,
        max_tokens=128000,
)

# %%
schema_linker = AssistantAgent(
            "schema_linker",
            model_client=schema_linker_client,
            description="SQL expert  extracting the most relevant tables and keys to the natural language query.",system_message=schema_linker_prompt,
)
sql_constructor = AssistantAgent(
            "sql_constructor",
            model_client=sql_constructor_client,
            description="SQL expert  constructing SQL query with given params.",
            system_message=sql_constructor_prompt,
)

join_expert = AssistantAgent(
            "join_expert",
            model_client=join_expert_client,
            description="SQL expert correcting JOINs in query if any error, removing unnecessary JOINs.",
            system_message=join_expert_prompt,
)

prompt_maker = AssistantAgent(
            "prompt_maker",
            model_client=tool_executor_client,
            tools=[prompt_maker_tool],
            description="Prompt maker generating the message using prompt_maker tool.",
            system_message=tool_client_prompt.format(tool="prompt_maker"),
)

syntax_fixer = AssistantAgent(
            "syntax_fixer",
            model_client=syntax_fixer_client,
            description="SQL expert fixing SQL query based on the error.",
            system_message="You are an AI assistant tasked with fixing SQL query given.",
)


In [ ]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_core.models import UserMessage

from spider_env import SpiderEnv
gym = SpiderEnv()


user_prompt =  """ 
### Task
Generate a SQL query to answer [QUESTION]{user_question}[/QUESTION]

### Database Schema
The query will run on a database with the following schema:
{table_metadata_string}

### Answer
Given the database schema, here is the SQL query
[SQL]
"""

participants=[schema_linker, sql_constructor, join_expert, prompt_maker, syntax_fixer]

for i in range(0, len(gym._dataset)):
    observation, info = gym.reset(k=i)
    g_dbid = observation["observation"]
    g_question = observation["instruction"]
    g_schema = info["schema"]

    data_team = RoundRobinGroupChat(participants=participants, termination_condition=None, max_turns=5)
    result = await Console(data_team.run_stream(task=user_prompt.format(user_question=g_question, table_metadata_string=g_schema)))
    final_sql=result.messages[-1].content.strip().replace("\n", "").replace("\t", "").replace("\n", "")
    
    await data_team.reset()
    # Define the user message
    with open("output_team.txt", "a") as file:
        file.write(f"{final_sql}\n")
    


### ADD OPENAI VECTOR STORE
